In [6]:
import sys
import os
import pickle
python_code_path = r"C:\Users\miioanni\Documents\PythonProjects\TrussGraph\codebase\src"
sys.path.append(python_code_path)
from HelperFunctions import Utilities
from Model import Graph

In [7]:
folder_path = r"C:\Users\miioanni\Documents\PythonProjects\TrussGraph\dataset\Pratt\p_6_seed_7"
graphs = Graph.ByJSONpath(folder_path)
print(f"Loaded {len(graphs)}x JSON files as Graph instances successfully.")

Loaded 4220x JSON files as Graph instances successfully.


In [8]:

import json
import torch
import torch.nn as nn
import torch.nn.functional as F

In [9]:
# ---------------------------------------------------------------------
# 1. Load JSON and build tensors
# ---------------------------------------------------------------------

def load_graph_info(graph):
    nodes = graph.VertexObjs
    n_node = graph.Order

    load_map = {n.Id: [0.0, 0.0, 0.0] for n in nodes}
    for ld in graph.LoadObjs:
        load_map[ld.NodeId] = [ld.X, ld.Y, ld.Z]
    
    X, Y = [], []
    for n in nodes:
        fx, fy, fz = load_map[n.Id]
        X.append([
            n.X, n.Y, n.Z,
            float(n.Label),
            float(n.Valency),
            fx, fy, fz
        ])
        Y.append([n.Ux, n.Uy, n.Uz])

    X = torch.tensor(X, dtype=torch.float32)
    Y = torch.tensor(Y, dtype=torch.float32)
    A = torch.zeros((n_node, n_node))
    for edge in graph.EdgeObjs:
        L = edge.Length
        A[edge.StartNode.Id][edge.EndNode.Id] = 1/L
        A[edge.EndNode.Id][edge.StartNode.Id] = 1/L
    
    return X, Y, A, graph

def normalize_adjacency(A):
    """Standard GCN renormalization trick: A_hat = D^-1/2 (A+I) D^-1/2."""
    n = A.shape[0]
    A_self = A + torch.eye(n)
    deg = A_self.sum(dim=1)
    d_inv_sqrt = torch.pow(deg, -0.5)
    D_inv_sqrt = torch.diag(d_inv_sqrt)
    return D_inv_sqrt @ A_self @ D_inv_sqrt

# ---------------------------------------------------------------------
# 2. A tiny 2-layer GCN
# ---------------------------------------------------------------------
class TrussGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim=1):
        super().__init__()
        self.W1 = nn.Linear(in_dim, hidden_dim)
        self.W2_1 = nn.Linear(hidden_dim, out_dim)
        self.W2_2 = nn.Linear(hidden_dim, out_dim)
        self.W2_3 = nn.Linear(hidden_dim, out_dim)
        
    def forward(self, X, A_hat):
        h = A_hat @ self.W1(X)
        h = F.relu(h)
        out1, out2, out3 = A_hat @ self.W2_1(h), A_hat @ self.W2_2(h), A_hat @ self.W2_3(h)
        return torch.cat([out1, out2, out3], dim=1)

In [10]:
for i in range(len(graphs)):
    graph = graphs[i]
    #graph = Utilities.ScaleResults(graphs[i], 300) # Scale to real-life scenarios
    #print(type(graph))
    X, Y, A, graph = load_graph_info(graph)
    #print(f"Graph {i}:")
    #print(f"X shape: {X.shape}")
    #print(X)
    #print(f"Y shape: {Y.shape}")
    #print(Y)
    #print(f"A shape: {A.shape}")
    #print(A)
    A_hat = normalize_adjacency(A)
    #print(f"A_hat shape: {A_hat.shape}")

    # Simple feature scaling helps the GCN converge.
    X_mean, X_std = X.mean(0, keepdim=True), X.std(0, keepdim=True) + 1e-6
    Xn = (X - X_mean) / X_std
    Y_mean, Y_std = Y.mean(0, keepdim=True), Y.std(0, keepdim=True) + 1e-6
    Yn = (Y - Y_mean) / Y_std

    model = TrussGCN(in_dim=X.shape[1], hidden_dim=32, out_dim=1)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)
    
    print(f"Graph: {graph.Order} nodes, {graph.Size} members")

    n_epochs = 1000
    for epoch in range(n_epochs):
        model.train()
        optimizer.zero_grad()
        pred = model(Xn, A_hat)
        loss = F.mse_loss(pred, Yn)
        loss.backward()
        optimizer.step()
        if epoch % 50 == 0 or epoch == n_epochs - 1:
            print(f"epoch {epoch:4d}  MSE (normalized) = {loss.item():.5f}")
    
    # De-normalize predictions back to physical units (mm-scale displacements)
    model.eval()
    with torch.no_grad():
        pred = model(Xn, A_hat) * Y_std + Y_mean
    
    print("\nnode_id |        true (Ux, Uy, Uz)        |      predicted (Ux, Uy, Uz)")
    print("-" * 78)
    for i, n in enumerate(graph.VertexObjs):
        t = Y[i].tolist()
        p = pred[i].tolist()
        print(f"{n.Id:7d} | ({t[0]:8.3f},{t[1]:6.3f},{t[2]:8.3f}) "
              f"| ({p[0]:8.3f},{p[1]:6.3f},{p[2]:8.3f})")

Graph: 12 nodes, 21 members
epoch    0  MSE (normalized) = 0.62292
epoch   50  MSE (normalized) = 0.19606


c:\Users\miioanni\Documents\PythonProjects\Autoencoders\vae\Lib\site-packages\torch\autograd\graph.py:882: UserWarning: cudaGetDeviceCount() returned cudaErrorNotSupported, likely using older driver or on CPU machine (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10\cuda\CUDAFunctions.cpp:88.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


epoch  100  MSE (normalized) = 0.07260
epoch  150  MSE (normalized) = 0.03764
epoch  200  MSE (normalized) = 0.03218
epoch  250  MSE (normalized) = 0.02855
epoch  300  MSE (normalized) = 0.02513
epoch  350  MSE (normalized) = 0.02172
epoch  400  MSE (normalized) = 0.01802
epoch  450  MSE (normalized) = 0.01469
epoch  500  MSE (normalized) = 0.01207
epoch  550  MSE (normalized) = 0.00944
epoch  600  MSE (normalized) = 0.00762
epoch  650  MSE (normalized) = 0.00628
epoch  700  MSE (normalized) = 0.00504
epoch  750  MSE (normalized) = 0.00420
epoch  800  MSE (normalized) = 0.00332
epoch  850  MSE (normalized) = 0.00355
epoch  900  MSE (normalized) = 0.00234
epoch  950  MSE (normalized) = 0.00202
epoch  999  MSE (normalized) = 0.00193

node_id |        true (Ux, Uy, Uz)        |      predicted (Ux, Uy, Uz)
------------------------------------------------------------------------------
      0 | (   0.000, 0.000,   0.000) | (  -0.563, 0.000,  -0.329)
      1 | ( -10.567, 0.000, -77.348) | ( 